In [1]:
#Import libraries

import pandas as pd

In [2]:
#Load retention
retention = pd.read_csv('retention_2024.csv')
admissions = pd.read_csv('admissions_2024.csv')
characteristics = pd.read_csv('characteristics_2024.csv')
financial_aid = pd.read_csv('financial_aid_2324.csv')
tuition = pd.read_csv('tuition_2024.csv')


In [3]:
#Merge datasets on UNITID
merged = pd.merge(retention, admissions, on='UNITID', how='inner')
merged = pd.merge(merged, characteristics, on='UNITID', how='inner')
merged = pd.merge(merged, financial_aid, on='UNITID', how='inner')
merged = pd.merge(merged, tuition, on='UNITID', how='inner')

merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1819 entries, 0 to 1818
Columns: 438 entries, UNITID to COTSFAM
dtypes: float64(131), int64(155), object(152)
memory usage: 6.1+ MB


In [4]:
#Narrow down to relevant columns for analysis
merged = merged[['UNITID', 'RET_PCF', 'RET_PCP', 'STUFACR', 'APPLCN', 'ADMSSN', 'ENRLT', 'ENRLFT', 
                 'ENRLPT', 'SATPCT', 'ACTPCT', 'CNTLAFFI', 'RELAFFIL', 'LEVEL5',
                 'OPENADMP', 'SLO6', 'SLO7', 'SLOA', 'DISABPCT',
                 'PGRNT_P', 'PGRNT_A', 'IGRNT_P', 'IGRNT_A', 'LOAN_P', 'LOAN_A', 'ANYAIDP',
                 'TUFEYR3', 'CINSON', 'COTSON']]

In [5]:
#Rename columns for clarity
merged = merged.rename(columns={
    'ACTPCT': 'act_submission_pct',
    'ADMSSN': 'admitted',
    'ANYAIDP': 'any_aid_pct',
    'APPLCN': 'applicants',
    'CINSON': 'coa_instate_oncampus',
    'CNTLAFFI': 'control_affiliation',
    'COTSON': 'coa_outofstate_oncampus',
    'DISABPCT': 'disability_pct',
    'ENRLFT': 'enrolled_fulltime',
    'ENRLPT': 'enrolled_parttime',
    'ENRLT': 'enrolled_total',
    'IGRNT_A': 'institutional_grant_avg',
    'IGRNT_P': 'institutional_grant_pct',
    'LEVEL5': 'offers_bachelors',
    'LOAN_A': 'loan_avg',
    'LOAN_P': 'loan_pct',
    'OPENADMP': 'open_admissions',
    'PGRNT_A': 'pell_grant_avg',
    'PGRNT_P': 'pell_grant_pct',
    'RELAFFIL': 'religious_affiliation',
    'RET_PCF': 'retention_fulltime',
    'RET_PCP': 'retention_parttime',
    'SATPCT': 'sat_submission_pct',
    'SLO6': 'study_abroad',
    'SLO7': 'evening_weekend',
    'SLOA': 'undergrad_research',
    'STUFACR': 'student_faculty_ratio',
    'TUFEYR3': 'tuition_fees',
    'UNITID': 'unit_id',
})

In [6]:
#Filter to 4-year, non-open admission, non-for-profit institutions
merged = merged[(merged['offers_bachelors'] == 1) & 
                (merged['open_admissions'] != 1) & 
                (merged['control_affiliation'] != 2)]

merged = merged.drop(columns=['offers_bachelors', 'open_admissions'])

In [7]:
#Encode religious affiliation as binary
merged['religious_affiliation'] = merged['religious_affiliation'].apply(lambda x: 0 if x == -2 else 1)

#One-hot encode control affiliation
merged = pd.get_dummies(merged, columns=['control_affiliation'], prefix='control', drop_first=True, dtype=int)

In [8]:
# Impute missing values with 0 for percentages and median for continuous variables
merged[['sat_submission_pct', 'act_submission_pct', 'disability_pct']] = merged[['sat_submission_pct', 'act_submission_pct', 'disability_pct']].fillna(0)

for col in ['pell_grant_avg', 'institutional_grant_avg', 'loan_avg', 
            'coa_instate_oncampus', 'coa_outofstate_oncampus']:
    merged[col] = merged[col].fillna(merged[col].median())

# Drop rows with missing admitted/enrolled values
merged = merged.dropna(subset=['admitted', 'enrolled_total'])


In [9]:
# Split into part-time and full-time datasets, dropping columns not relevant to each.
fulltime = merged.dropna(subset=['retention_fulltime']).drop(columns=['retention_parttime', 'enrolled_parttime'])
parttime = merged.dropna(subset=['retention_parttime']).drop(columns=['retention_fulltime', 'enrolled_fulltime'])

# Drop df-specific missing values
fulltime = fulltime.dropna(subset=['enrolled_fulltime'])
parttime = parttime.dropna(subset=['enrolled_parttime', 'pell_grant_pct', 'institutional_grant_pct', 'any_aid_pct'])

In [10]:
#output cleaned datasets
fulltime.to_csv('fulltime_clean.csv', index=False)
parttime.to_csv('parttime_clean.csv', index=False)